# Rich Neale's Blocking Diagnostic (ESNB version)
More details on the development process:
[MDTF Planning Document](https://docs.google.com/document/d/1P8HqL8O5304qwR3ik9RmgFDwSWwlkPgOjnp39PIkLfY/edit?usp=sharing)

[ESNB GitHub Repository](https://github.com/jkrasting/esnb/tree/ncar-updates)

In [ ]:
# Import General Package

import importlib

import sys
import yaml
import json

import os


: 

General Settings


In [ ]:


# Give your diagnostic a name and a short description
diag_name = "Blocking (Neale) Notebook-based Diagnostic"
diag_desc = "This example demonstrates how to use the notebook template"


In [ ]:
#
# Settings for how this Notebook is being used
# 

mode = "interactive"
# mode:
#        interactive = running this as a notebook, input is provided through the notebook interface
#        driver      = (Not yet available) MDTF called from the command line, input provided through command line arguments (-f file.yaml)

usage = "dev"
# usage:
#         dev  (development): either CESM time series files with no pre-processing done
#                             OR pre-processed files already made, used directly
#         prod (production) : MDTF full functionality. 
#                             Any supported data type is provided through catalogs and translated as necessary
#                             by the MDTF pre-processor.
#       

machine = "casper"
# machine:
#         Uses machine-specfic things such as dask. 
#         This should be determined by the framework
#       

test = 0 
# test: 
#         test = 1: find and load data, skip analysis/plotting
#         test = 0: run the full analysis and plottings
#

# Verbosity
verbose = True



In [ ]:
# Development usage: constantly refreshes module code

if (usage == "dev"):  
    %load_ext autoreload
    %autoreload 2


In [ ]:
#
# Machine Specific Settings
# These are particular to NCAR's Casper HPC system. For other systems, add your own settings here.
#

if (machine ==  "casper"):
    print("Using Casper-specific settings")
    # Use dask jobqueue
    from dask_jobqueue import PBSCluster

    # Import a client
    from dask.distributed import Client

    #P03010039. #P93300642

    cluster = PBSCluster(cores=4, memory="8GB", account='P03010039', interface='ext', job_extra_directives=[], walltime='2:00:00', queue='casper', resource_spec='select=1:ncpus=4:mem=8GB')
    cluster
    cluster.scale(jobs=1)
    client = Client(cluster)
    client

#Ideally this then waits to confirm the job has started, since it goes through the queuing system. Current usage: use qstat -u $USER in the terminal to find it.

## Framework Code and Diagnostic Setup

In [ ]:
# 
# Framework: Setup paths to import MDTF code as modules
#
module_path = os.path.abspath(os.path.join('../../'))  #Assumes the notebook is in MDTF-diagnostics/diagnostics/diagnostic_name/notebook_name
print(f"Using MDTF framework version {module_path}")
if module_path not in sys.path:
    sys.path.append(module_path)
from src import util              #, varlist_util, translation, xr_parser, units
from src.util import json_utils

import blocking_utils as block_utils
import blocking_figs as block_figs


In [ ]:
# The ESNB package has  notebook-specific classes and functions
import esnb
from esnb import NotebookDiagnostic, RequestedVariable, CaseGroup2

os.environ["ESNB_LOG_LEVEL"] = "DEBUG"

In [ ]:
#### POD settings are defined here (interactive mode) or read
#### from the diagnostics/blocking_notebook/settings.jsonc file  

if (mode == "interactive"):
  settings_dict = {
    "settings": {

        # ---------------------------------------------------
        #The pod_env_vars are used in the POD, not required by the MDTF framework
        # ---------------------------------------------------
        "pod_env_vars": {  
            "MDTF_BLOCKING_OBS": True,
            "MDTF_BLOCKING_OBS_USE_CASE_YEARS": False,
            "MDTF_BLOCKING_OBS_ERA_FIRSTYR": 2010,
            "MDTF_BLOCKING_OBS_ERA_LASTYR": 2014,
            "MDTF_BLOCKING_OBS_MERRA_FIRSTYR": 2009,
            "MDTF_BLOCKING_OBS_MERRA_LASTYR": 2011,
            "MDTF_BLOCKING_OBS_CAM5_FIRSTYR": 1979,
            "MDTF_BLOCKING_OBS_CAM5_LASTYR": 2007,
            "MDTF_BLOCKING_CAM3": False,
            "MDTF_BLOCKING_CAM4": False,
            "MDTF_BLOCKING_CAM5": True,
            "MDTF_BLOCKING_READ_DIGESTED": True,
            "MDTF_BLOCKING_WRITE_DIGESTED": False,
            "MDTF_BLOCKING_WRITE_DIGESTED_DIR": "",
            "MDTF_BLOCKING_DEBUG": False
        },
        
        # ---------------------------------------------------
        # The following settings are used by the MDTF framework
        # ---------------------------------------------------
        "runtime_requirements": {
            "python3": None,  # None means use the default python version of the MDTF framework
        },
        "driver": "blocking_main.ipynb",
        "long_name": "Rich Neale's blocking diagnostic",
        "convention": "cesm",
        "description": "Rich Neale's blocking diagnostic", 
    },
    "dimensions": {
        "lat": {
            "standard_name": "latitude",
            "units": "degrees_north",
            "axis": "Y"
        },
        "lon": {
            "standard_name": "longitude",
            "units": "degrees_east",
            "axis": "X"
        },
        "lev": {
            "standard_name": "air_pressure",
            "units": "hPa",
            "positive": "down",
            "axis": "Z"
        },
        "time": {
            "standard_name": "time"
        }
    },
    "varlist": {
        "zg": {
            "path_variable": "MODEL_DATA_PATH",  
            "standard_name": "geopotential_height",
            "units": "m",
            "realm": "atmos",
            "frequency": "day",
            "dimensions": [
                "time",
                "lat",
                "lon"
            ],
            "scalar_coordinates": {
                "lev": 500
            }
        }
    }
}
else:  # batch mode: running from the command line, read in the settings file
    import json

    # Specify the path to the JSONC file
    settings_file_path = "./settings.jsonc"

    try:
        # Open and read the JSONC file using MDTF framework's json_utils
        settings = json_utils.read_json(settings_file_path)
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Pretty-print the loaded JSON data
print(json_utils.pretty_print_json(settings_dict))
    



In [ ]:
# We should ask developers to generate a settings file before submitting. But while developing, it is easier to use the settings_dict directly.

#  Initialize the diagnostic with its name, description, vars, and options
# This is the setttings file from mdtf/diagnostics/$DIAG_NAME/settings.jsonc (Must be full path)
#test diag = NotebookDiagnostic("/glade/u/home/bundy/diag/mdtf/esnb/esnb/src/esnb/data/settings.jsonc")
diag = NotebookDiagnostic(settings_dict)
#print(diag)
from pprint import pprint
pprint(vars(diag))

In [ ]:
#Variable name, and possibly other information. This could be deduced from the mdtf/data/fieldlist_$MODEL.json files
mapping = {
    "zg": "Z500",
}


In [ ]:

if (mode == "interactive"):
    # If running in interactive mode, input case settings are made here. Otherwise they are read from the input settings file
    # This should contain everything in the input file. Extra key value pairs can be added in a later cell.
    case_info = {
     "case_list": {
            "timeslice_mdtfv3": { #case name
                "convention": "CESM",
                "enddate": "19961231000000",
                "model": "CESM",
                "startdate": "19950101000000"
            }
        },
        "DATA_CATALOG": "/glade/u/home/bundy/diag/mdtf/catalogs/esm_catalog_CESM_timeslice_mdtfv3.20241107.json",
        "OBS_DATA_ROOT": "/glade/work/bundy/mdtf/inputdata/obs_data",
        "WORK_DIR": "/glade/work/bundy/mdtf/outdir/20250429/blocking_nb",
        "OUTPUT_DIR": "/glade/work/bundy/mdtf/outdir/20250429/blocking_nb", #leaving blank will use the WORK_DIR
        "conda_env_root": "/glade/work/bundy/miniconda2/envs.MDTF.20241107/",
        "conda_root": "/glade/work/bundy/miniconda2/",
        "large_file": False,
        "make_multicase_figure_html": False,
        "make_variab_tar": False,
        "micromamba_exe": "",
        "overwrite": True,
####       "pod_list": ["Wheeler_Kiladis"],
        "run_pp": True,
        "save_pp_data": True,
        "save_ps": False,
        "translate_data": True,
        "user_pp_scripts": [""]
    }
else:
    # Receive a dictionary of case information from the framework. 
    # This needs to be passed from the MDTF framework (will be an argument to the command line call)"
    case_input_file =  "../../input_files/input_timeslice_test.yml"
    print("reading default settings from {case_input_file}")

    assert os.path.isfile(case_input_file), f"case environment file not found"
    with open(case_input_file, 'r') as stream:
        try:
            case_info = yaml.safe_load(stream)
        except yaml.YAMLError as exc:
            print(exc)

print(yaml.dump(case_info))



In [ ]:
# 
# Give the case info to the ESNB CaseGroup2 class 
#
mdtf_groups = [
    CaseGroup2(
        case_info,
        date_range=("1995-01-01", "1996-12-31"),
        mapping=mapping,
    )
]

In [ ]:
#Inspect the one case more carefully
mdtf_groups[0].cases[0]

In [ ]:
#
# ESNB Function Read the catalog, rename variables, find files to open
diag.resolve(mdtf_groups)

In [ ]:
# Inspect which files are actually being read (as found through the catalog)
diag.files

In [ ]:
#Open the model data files using ESNB function
diag.open()

In [ ]:
# Show what datasets are available
diag.datasets

<i>(The files above are necessary to run the diagnostic.)</i>

In [ ]:
diag.variables[0]

## Example: Exploring Datasets by Looping Over Variables

In [ ]:
# First loop over variables, and then over groups

for variable in diag.variables:
    for group in variable.datasets.keys():
        ds = variable.datasets[group]
        print("\n")
        print(f"Variable={variable}, Group={group}")
        print(ds)

In [ ]:
#See some actual numbers, just checking that the data is there
#print(ds["Z500"].values[:10, :10, :10])  # Print first 10 values for Z500 variable

## Example: Exploring Datasets by Looping over Groups

In [ ]:
# First loop over groups, and then over variables

for group in diag.groups:
    for variable in group.datasets.keys():
        ds = group.datasets[variable]
        print("\n")
        print(f"Variable={variable}, Group={group}")
        print(ds)

# Blocking Code Main Routines


### Load OBS data 

In [ ]:

#### ...Ensemble comparison cases...

ensemble_configs = {
    "CESM1": {
        "ens_type": "model",
        "dir_ens0": "/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/",
        "dir_day_add": "daily",
        "file_template": lambda run: f"/glade/campaign/cesm/collections/cesmLE/CESM-CAM5-BGC-LE/atm/proc/tseries/daily/VAR_TBD/{run}.cam.h1.VAR_TBD.19200101-20051231.nc",
        "special_case": lambda run, file: file.replace('1920', '1850', 1) if run == 'b.e11.B20TRC5CNBDRD.f09_g16.001' else file
    },
    "CESM2": {
        "ens_type": "model",
        "dir_ens0": "/glade/campaign/cgd/cesm/CESM2-LE/atm/proc/tseries/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/cesm/CESM2-LE/atm/proc/tseries/day_1/VAR_TBD/{run}.cam.h1.VAR_TBD.DATE_RANGE.nc"
    },
    "E3SMv1": {
        "ens_type": "model",
        "dir_ens0": "/glade/campaign/cgd/amp/rneale/e3sm/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/amp/rneale/e3sm/{run}/tseries/{run}_dmeans_ts_VAR_TBD.nc"
    },
    "E3SMv2": {
        "ens_type": "model",
        "cmodel": "eam",
        "dir_ens0": "/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/{run}/atm/proc/tseries/day_1/{run}.eam.h1.VAR_TBD.18500101-20141231.nc"
    },
    "EAMv2": {
        "ens_type": "model",
        "cmodel": "eam",
        "dir_ens0": "/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cgd/ccr/E3SMv2/FV_regridded/{run}/atm/proc/tseries/day_1/{run}.eam.h1.VAR_TBD.19760101-20141231.nc"
    },
    "CAM6": {
        "ens_type": "model",
        "cmodel": "cam",
        "dir_ens0": "/glade/campaign/cesm/development/cvcwg/cvwg/f.e21.FHIST_FSSP370_BGC.f09_f09.ersstv5.goga/",
        "dir_day_add": "day_1",
        "file_template": lambda run: f"/glade/campaign/cesm/development/cvcwg/cvwg/f.e21.FHIST_FSSP370_BGC.f09_f09.ersstv5.goga/{run}/atm/proc/tseries/day_1/{run}.cam.h1.VAR_TBD.18800101-20150101.nc"
    },
    "b.e23": {
        "ens_type": "model",
        "dir_ens0": "/glade/derecho/scratch/hannay/archive/",
        "file_template": lambda run: f"/glade/derecho/scratch/hannay/archive/{run}/atm/hist/{run}.cam.h1..nc"
    },
    # Add obs sources dynamically
}

# Example obs_sources list
obs_sources = ['ERA5', 'MERRA', 'CESM2']

for obs in obs_sources:
    ensemble_configs[obs] = {
        "ens_type": "obs",
        "dir_ens0": f"/glade/work/rneale/data/{obs}/",
        "file_template": lambda run=None: f"/glade/work/rneale/data/{obs}/VAR_TBD.day.mean.nc"
    }

# Usage example:
ens_name = "CESM1"
run_names = ["b.e11.B20TRC5CNBDRD.f09_g16.001", "b.e11.B20TRC5CNBDRD.f09_g16.002"]
config = ensemble_configs[ens_name]
file_templates = [config["file_template"](run) for run in run_names]
if "special_case" in config:
    file_templates = [config["special_case"](run, file) for run, file in zip(run_names, file_templates)]

print(file_templates)
# ...existing code...



In [ ]:
print(ensemble_configs)

In [ ]:
### DRBDBG: This is from Rich's original, but I think it should work here. 
### Check if it repeats what is done above, and/or is used at all!
###       Eventually pull the ens_setup to the cell level though so user can modify

### Read in obs/ens data
### block_meta type: <class 'pandas.core.frame.DataFrame'>
### block_meta columns:
### ['Ensemble Type', 'Ensemble Size', 'Start Year', 'End Year', 'Run Name', 'Run File'],
### block_meta index: Index(['ERA5', 'MERRA', 'CESM2'], dtype='object')

start_year = '1990' #1979
end_year = '1991' #2005
importlib.reload(block_utils) # Update for any edits to utility routines.
ens_names = ['ERA5','MERRA','CESM2']
ens_mem_num = [1,1,2]  # Ensemble members per ensemble set (testing version with just a few ensemble members)
ens_ystart = [start_year]*len(ens_names)
ens_yend     = [end_year]*len(ens_names)
block_season = 'DJF'
cam_var = 'Z500' #DRBDBG where are the obs variable names set?
block_diag_hem = ['nhem']  
block_diag_set = ['1d']

# Grab basic ens+run information and populate a dataframe
block_meta = block_utils.ens_setup(ens_names,ens_mem_num,ens_ystart,ens_yend)

In [ ]:
#See what is in the block_meta structure

print("block_meta type:", type(block_meta))
print("block_meta columns:", block_meta.columns if hasattr(block_meta, 'columns') else None)
print("block_meta index:", block_meta.index if hasattr(block_meta, 'index') else None)
print("block_meta head:")
print(block_meta.head())
print("block_meta info:")
if hasattr(block_meta, 'info'):
    block_meta.info()

In [ ]:

print(f"\n Reading files from table below")
fout_dir = case_info['OUTPUT_DIR']
print(f"\n Writing output to {fout_dir}")

# Grab basic ens+run information and populate a dictionary)
block_meta = block_utils.ens_setup(ens_names,ens_mem_num,ens_ystart,ens_yend)

print(f"block_meta keys: {block_meta.keys()}")
print(f"block_meta: {block_meta}")


### Extract Model (new case(s)) data from structure

In [ ]:
# Extract the MDTF group/case information and add to the pandas structure
import pandas as pd

for group in diag.groups: 
    print(f"Group: {group.name}")
    n_cases = len(group.datasets)
    print(f"Found {n_cases=} in group")
    print(f"  Group items {group.cases=}")

    for case in group.cases:
        print(f"Case attributes for {case.name}:")
        #for attr, value in vars(case).items():
        #fails on Nans in realm column
        #    print(f"  {attr}: {value}")
            
        print(f"    name: {case.name}")
        print(f"    Files: {case.files()}")
        print(f"    source: {case.source}")        
        #print(f"    all mdtf_settings: {case.mdtf_settings}")
        
        #Example on how to get something out of the structure
        mdtf_settings = case.mdtf_settings
        
        #All settings 
        case_list = mdtf_settings.get('case_list')
        print(f"    {case_list=}") 
        
        for case_name, case_details in case_list.items():
            print(f"Model: {case_details.get('model')}")
            print(f"Convention: {case_details.get('convention')}")
            print(f"Start Date: {case_details.get('startdate')}")
            print(f"End Date: {case_details.get('enddate')}")


    # Add to the dictionary
    new_row = {
        'Ensemble Type': 'mdtf',
        'Ensemble Size': n_cases,
        'Start Year': case_details['startdate'], #these are all the same for the group
        'End Year': case_details['enddate'],     #these are all the same for the group
        'Run Name': case_name, #need to do multiple 
        'Run File': diag.files #how does this work with the multiple cases, even groups??
        # could pass xarrays instead of file names: diag.datasets[group.name]   
        }
    new_row_df = pd.DataFrame([new_row], index=[case_name])
    #print(new_row_df)

    block_meta = pd.concat([block_meta, new_row_df])
    

print(block_meta)




In [ ]:
block_season = 'DJF'
block_diag_hem = ['nhem']  
block_diag_set = ['1d']



# Grab daily datasets needed for each 'ensemble'

In [ ]:
cam_var = 'Z500'  #The name of the variable in the digested data, not necessarily the variable in the catalog
ensemble_ds = block_utils.dataset_get(block_meta,cam_var,block_season,block_diag_hem)

In [ ]:
print(f"ensemble_ds: {ensemble_ds}")

#####  Calculate/Read/Write blocking frequency: 1D

In [ ]:
#print(ensemble_ds)
ens_names = list(block_meta.index)
print(f"Ensemble names: {ens_names}")

#i = 1
#ens_name = ens_names[i]
#print(f"Changing name of ensemble  {ens_name=}")
#ensemble_ds[ens_name].coords["name"] = ens_name


for i in range(len(ens_names)):
    ens_name = ens_names[i]
    print(f"++++ Ensemble {i}: {ens_name}")
    print(f"      {ensemble_ds[ens_name]}")


#one_ds = 
#print(f"one_ds: {one_ds}")


In [ ]:
# Either calculate and write a file with, or read in the blocking frequency
importlib.reload(block_utils) # Update for any edits to utility routines.

ensemble_block_1d = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,block_season,block_diag = '1D', file_opts = 'x')


##### Calculate/Read/Write blocking frequency: 2D


In [ ]:
ensemble_block_2d = block_utils.block_z500_freq(block_meta,ensemble_ds,fout_dir,block_season,block_diag = '2D', file_opts = 'x')


In [ ]:
block_figs.block_plot_1d(block_meta,ensemble_block_1d,block_season,fig_out=True,dir_fig=fout_dir)



##### Plot 2D blocking: Ensemble ave, 1 ensemble member, or all ensemble members 


In [ ]:
block_figs.block_plot_2d(block_meta,ensemble_block_2d,block_season,fig_out=True,dir_fig=fout_dir,ens_plot='av')

In [ ]:
block_figs.block_plot_1d(block_meta,ensemble_block_1d,block_season,fig_out=True,dir_fig=fout_dir)